In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import recall_score, classification_report, confusion_matrix
from tqdm import tqdm

print("Imports OK")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Imports OK
Device: NVIDIA GeForce RTX 2050


In [2]:
data = np.load(r'data/icbhi_preprocessed.npz')

X_train       = data['mel_train']
X_test        = data['mel_test']
y_train       = data['y_train']
y_test        = data['y_test']
class_weights = data['class_weights']

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("class_weights:", class_weights)

X_train: (10353, 128, 251)
X_test : (2756, 128, 251)
y_train: (10353,)
y_test : (2756,)
class_weights: [1.2582644 0.5347624 1.2915419 1.7825413]


In [3]:
class ICBHIDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, 128, 251)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = ICBHIDataset(X_train, y_train)
test_dataset  = ICBHIDataset(X_test,  y_test)

# WeightedRandomSampler — rééquilibre les classes rares
class_counts      = np.bincount(y_train)
class_w_sampler   = 1. / class_counts
sample_weights    = class_w_sampler[y_train]

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(sample_weights),
    replacement=True
)

# shuffle=False car le sampler gère déjà le mélange
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 324
Test batches  : 87


In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=5, padding=2, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)

    def forward(self, x, pool_size=(2, 2)):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.avg_pool2d(x, pool_size)
        return x

class CNN6Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.bn0         = nn.BatchNorm2d(64)
        self.conv_block1 = ConvBlock(1,   64)
        self.conv_block2 = ConvBlock(64,  128)
        self.conv_block3 = ConvBlock(128, 256)
        self.conv_block4 = ConvBlock(256, 512)
        self.fc1         = nn.Linear(512, 512)
        self.fc_audioset = nn.Linear(512, 527)

    def forward(self, x):
        # x : (batch, 1, 128, 251)
        x = self.conv_block1(x, pool_size=(2, 2))  # (batch, 64,  64, 125)
        x = self.conv_block2(x, pool_size=(2, 2))  # (batch, 128, 32,  62)
        x = self.conv_block3(x, pool_size=(2, 2))  # (batch, 256, 16,  31)
        x = self.conv_block4(x, pool_size=(2, 2))  # (batch, 512,  8,  15)
        x = torch.mean(x, dim=3)                   # (batch, 512,   8)
        x, _ = torch.max(x, dim=2)                 # (batch, 512)
        x = F.relu(self.fc1(x))                    # (batch, 512)
        return x

class CNN6Classifier(nn.Module):
    def __init__(self, num_classes=4,
                 checkpoint_path='panns_weights/Cnn6_mAP=0.343.pth'):
        super().__init__()
        self.backbone = CNN6Backbone()

        checkpoint = torch.load(checkpoint_path, map_location='cpu',
                                weights_only=False)
        missing, unexpected = self.backbone.load_state_dict(
            checkpoint['model'], strict=False
        )
        print(f"Clés manquantes  : {missing}")
        print(f"Clés inattendues : {len(unexpected)} clés ignorées ")
        print("Poids CNN6 chargés ")

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x : (batch, 128, 251)
        x = x.unsqueeze(1)           # (batch, 1, 128, 251)
        features = self.backbone(x)  # (batch, 512)
        return self.classifier(features)

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cnn6 = CNN6Classifier(num_classes=4).to(device)
print(f"CNN6 chargé sur : {device} ")

Clés manquantes  : []
Clés inattendues : 3 clés ignorées 
Poids CNN6 chargés 
CNN6 chargé sur : cuda 


In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.alpha,
            reduction='none'
        )
        pt          = torch.exp(-ce_loss)
        focal_loss  = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

print("Focal Loss prête ")

Focal Loss prête 


In [6]:
weights   = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Focal Loss avec class weights de Bochra
criterion = FocalLoss(alpha=weights, gamma=2)

# Fine-tuning progressif
optimizer = torch.optim.AdamW([
    {'params': model_cnn6.backbone.parameters(),   'lr': 1e-5},
    {'params': model_cnn6.classifier.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print("Loss, Optimizer et Scheduler prêts ")

Loss, Optimizer et Scheduler prêts 


In [7]:
def compute_icbhi_metrics(all_labels, all_preds):
    # Normal = 0, Anormal = 1/2/3
    binary_true = [0 if y == 0 else 1 for y in all_labels]
    binary_pred = [0 if y == 0 else 1 for y in all_preds]

    cm_binary        = confusion_matrix(binary_true, binary_pred)
    TN, FP, FN, TP   = cm_binary.ravel()

    sensitivity  = TP / (TP + FN)
    specificity  = TN / (TN + FP)
    icbhi_score  = (sensitivity + specificity) / 2

    return sensitivity, specificity, icbhi_score

def evaluate(model, loader, device):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds   = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    recall                              = recall_score(all_labels, all_preds, average='macro')
    cm                                  = confusion_matrix(all_labels, all_preds)
    sensitivity, specificity, icbhi     = compute_icbhi_metrics(all_labels, all_preds)

    return recall, cm, all_preds, all_labels, sensitivity, specificity, icbhi

print("Métriques ICBHI prêtes ")

Métriques ICBHI prêtes 


In [8]:
num_epochs  = 20
best_score  = 0   # on sauvegarde sur ICBHI score
history     = {'loss': [], 'recall': [], 'icbhi': []}

for epoch in range(num_epochs):
    model_cnn6.train()
    total_loss = 0
    correct    = 0
    total      = 0

    loop = tqdm(train_loader,
                desc=f"Epoch {epoch+1}/{num_epochs}",
                leave=True)

    for X_batch, y_batch in loop:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model_cnn6(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs, dim=1)
        correct    += (preds == y_batch).sum().item()
        total      += y_batch.size(0)

        loop.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc' : f'{correct/total:.2%}'
        })

    scheduler.step()

    recall, cm, all_preds, all_labels, sensitivity, specificity, icbhi = evaluate(
        model_cnn6, test_loader, device
    )
    avg_loss = total_loss / len(train_loader)

    history['loss'].append(avg_loss)
    history['recall'].append(recall)
    history['icbhi'].append(icbhi)

    print(f"\n Epoch {epoch+1}/{num_epochs}")
    print(f"   Loss            : {avg_loss:.4f}")
    print(f"   Recall macro    : {recall:.4f}")
    print(f"   Sensitivity (Se): {sensitivity:.4f}")
    print(f"   Specificity (Sp): {specificity:.4f}")
    print(f"   ICBHI Score     : {icbhi:.4f}  (baseline: 0.6810)")
    print(f"   Meilleur ICBHI  : {best_score:.4f}")
    print(f"   Matrice de confusion :\n{cm}")
    print("-" * 50)

    # Sauvegarder sur ICBHI score
    if icbhi > best_score:
        best_score = icbhi
        torch.save(model_cnn6.state_dict(), 'best_cnn6.pth')
        print(f"    Nouveau meilleur modèle sauvegardé ! (ICBHI: {best_score:.4f})")

print(f"\n Entraînement terminé !")
print(f"   Meilleur ICBHI Score : {best_score:.4f}")
print(f"   Baseline à battre   : 0.6810")

Epoch 1/20: 100%|██████████| 324/324 [58:45<00:00, 10.88s/it, loss=0.7313, acc=25.79%]  



 Epoch 1/20
   Loss            : 0.7601
   Recall macro    : 0.3075
   Sensitivity (Se): 0.4860
   Specificity (Sp): 0.5332
   ICBHI Score     : 0.5096  (baseline: 0.6810)
   Meilleur ICBHI  : 0.0000
   Matrice de confusion :
[[842  19 147 571]
 [390  24   7 228]
 [158  10  36 181]
 [ 57   3   2  81]]
--------------------------------------------------
    Nouveau meilleur modèle sauvegardé ! (ICBHI: 0.5096)


Epoch 2/20: 100%|██████████| 324/324 [48:54<00:00,  9.06s/it, loss=0.5460, acc=38.21%] 



 Epoch 2/20
   Loss            : 0.5928
   Recall macro    : 0.3328
   Sensitivity (Se): 0.3356
   Specificity (Sp): 0.6681
   ICBHI Score     : 0.5019  (baseline: 0.6810)
   Meilleur ICBHI  : 0.5096
   Matrice de confusion :
[[1055   49  304  171]
 [ 529   34   18   68]
 [ 180    8  114   83]
 [  73    3   22   45]]
--------------------------------------------------


Epoch 3/20: 100%|██████████| 324/324 [03:16<00:00,  1.65it/s, loss=0.2436, acc=45.82%]


...


 Epoch 18/20
   Loss            : 0.3090
   Recall macro    : 0.3496
   Sensitivity (Se): 0.3500
   Specificity (Sp): 0.7175
   ICBHI Score     : 0.5338  (baseline: 0.6810)
   Meilleur ICBHI  : 0.5408
   Matrice de confusion :
[[1133   53  259  134]
 [ 525   51   33   40]
 [ 174    4  100  107]
 [  66    3   25   49]]
--------------------------------------------------


Epoch 19/20: 100%|██████████| 324/324 [01:41<00:00,  3.18it/s, loss=0.5033, acc=64.69%]



 Epoch 19/20
   Loss            : 0.3107
   Recall macro    : 0.3541
   Sensitivity (Se): 0.3840
   Specificity (Sp): 0.7011
   ICBHI Score     : 0.5426  (baseline: 0.6810)
   Meilleur ICBHI  : 0.5408
   Matrice de confusion :
[[1107   63  273  136]
 [ 502   58   34   55]
 [ 163    5  109  108]
 [  60    3   31   49]]
--------------------------------------------------
    Nouveau meilleur modèle sauvegardé ! (ICBHI: 0.5426)


Epoch 20/20: 100%|██████████| 324/324 [02:10<00:00,  2.49it/s, loss=0.3076, acc=64.39%]



 Epoch 20/20
   Loss            : 0.3095
   Recall macro    : 0.3531
   Sensitivity (Se): 0.3806
   Specificity (Sp): 0.6922
   ICBHI Score     : 0.5364  (baseline: 0.6810)
   Meilleur ICBHI  : 0.5426
   Matrice de confusion :
[[1093   75  260  151]
 [ 499   70   26   54]
 [ 167    7   93  118]
 [  63    4   23   53]]
--------------------------------------------------

 Entraînement terminé !
   Meilleur ICBHI Score : 0.5426
   Baseline à battre   : 0.6810


In [9]:
model_cnn6.load_state_dict(torch.load('best_cnn6.pth'))
recall, cm, all_preds, all_labels, sensitivity, specificity, icbhi = evaluate(
    model_cnn6, test_loader, device
)

print(" Rapport final CNN6 :")
print(classification_report(
    all_labels, all_preds,
    target_names=['Normal', 'Crackle', 'Wheeze', 'Both']
))
print(f"Recall macro     : {recall:.4f}")
print(f"Sensitivity (Se) : {sensitivity:.4f}  (baseline: 0.6831)")
print(f"Specificity (Sp) : {specificity:.4f}  (baseline: 0.6789)")
print(f"ICBHI Score      : {icbhi:.4f}  (baseline: 0.6810)")

C:\Users\Ons Souidi\AppData\Local\Temp\ipykernel_8984\1560725679.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_cnn6.load_state_dict(torch.load('best_cnn6.pth'))


 Rapport final CNN6 :
              precision    recall  f1-score   support

      Normal       0.60      0.70      0.65      1579
     Crackle       0.45      0.09      0.15       649
      Wheeze       0.24      0.28      0.26       385
        Both       0.14      0.34      0.20       143

    accuracy                           0.48      2756
   macro avg       0.36      0.35      0.31      2756
weighted avg       0.49      0.48      0.45      2756

Recall macro     : 0.3541
Sensitivity (Se) : 0.3840  (baseline: 0.6831)
Specificity (Sp) : 0.7011  (baseline: 0.6789)
ICBHI Score      : 0.5426  (baseline: 0.6810)
